In [0]:
# ============================================================
# Ajay | Notebook 2: Raw File → Bronze Delta Table
# ============================================================

dbutils.widgets.text("catalog", "dev_team")
dbutils.widgets.text("schema",  "testing")
CATALOG       = dbutils.widgets.get("catalog")
SCHEMA        = dbutils.widgets.get("schema")
VOLUME        = "files_volume"
BASE_PATH     = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/Ajay/ajay_flights_vol"
BRONZE_TABLE  = f"`{CATALOG}`.`{SCHEMA}`.Ajay_bronze_flight_raw"

print(f"✅ Config loaded")
print(f"   Bronze Table : {BRONZE_TABLE}")
print(f"   Source Path  : {BASE_PATH}")

✅ Config loaded
   Bronze Table : `dev_team`.`testing`.Ajay_bronze_flight_raw
   Source Path  : /Volumes/dev_team/testing/files_volume/Ajay/ajay_flights_vol


In [0]:
# Get the most recent JSON file from the Volume
files = dbutils.fs.ls(BASE_PATH)

json_files = sorted(
    [f for f in files if f.name.endswith(".json")],
    key=lambda x: x.name,
    reverse=True
)

if not json_files:
    raise Exception(f"❌ No JSON files found in {BASE_PATH}")

raw_file_path = json_files[0].path
raw_file_name = json_files[0].name

print(f"✅ Latest file found!")
print(f"   📄 File name : {raw_file_name}")
print(f"   📁 Full path : {raw_file_path}")
print(f"   📊 Total JSON files in volume : {len(json_files)}")

✅ Latest file found!
   📄 File name : ajay_flights_20260505_174937.json
   📁 Full path : dbfs:/Volumes/dev_team/testing/files_volume/Ajay/ajay_flights_vol/ajay_flights_20260505_174937.json
   📊 Total JSON files in volume : 1


In [0]:
from pyspark.sql import functions as F

# Read the raw JSON file into Spark
print(f"📖 Reading raw JSON file...")

raw_df = spark.read \
    .option("multiline", "true") \
    .json(raw_file_path)

print(f"✅ File loaded into Spark DataFrame!")
print(f"   📊 Rows    : {raw_df.count()}")
print(f"   📋 Columns : {len(raw_df.columns)}")
print(f"\n🔍 Schema:")
raw_df.printSchema()

📖 Reading raw JSON file...
✅ File loaded into Spark DataFrame!
   📊 Rows    : 1
   📋 Columns : 2

🔍 Schema:
root
 |-- states: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: string (containsNull = true)
 |-- time: long (nullable = true)



In [0]:
# Add audit/metadata columns to raw data
bronze_df = raw_df \
    .withColumn("source_file",  F.lit(raw_file_path)) \
    .withColumn("insert_dttm",  F.current_timestamp()) \
    .withColumn("update_dttm",  F.current_timestamp()) \
    .withColumn("inserted_by",  F.current_user()) \
    .withColumn("updated_by",   F.current_user())

print(f"✅ Audit columns added!")
print(f"   Columns now : {len(bronze_df.columns)}")
print(f"\n🔍 Final Bronze Schema:")
bronze_df.printSchema()

✅ Audit columns added!
   Columns now : 7

🔍 Final Bronze Schema:
root
 |-- states: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: string (containsNull = true)
 |-- time: long (nullable = true)
 |-- source_file: string (nullable = false)
 |-- insert_dttm: timestamp (nullable = false)
 |-- update_dttm: timestamp (nullable = false)
 |-- inserted_by: string (nullable = false)
 |-- updated_by: string (nullable = false)



In [0]:
# Write to Bronze Delta table
print(f"💾 Writing to Bronze table: {BRONZE_TABLE}")

bronze_df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(BRONZE_TABLE)

row_count = spark.table(BRONZE_TABLE).count()

print(f"✅ Bronze table written successfully!")
print(f"   📋 Table     : {BRONZE_TABLE}")
print(f"   📊 Row count : {row_count}")

💾 Writing to Bronze table: `dev_team`.`testing`.Ajay_bronze_flight_raw
✅ Bronze table written successfully!
   📋 Table     : `dev_team`.`testing`.Ajay_bronze_flight_raw
   📊 Row count : 1


In [0]:
# Preview the Bronze table
print(f"🔍 Bronze Table Preview:")

spark.table(BRONZE_TABLE).select(
    "time",
    "insert_dttm",
    "inserted_by",
    "source_file"
).show(5, truncate=60)

print(f"\n📊 Quick stats:")
print(f"   Total rows    : {spark.table(BRONZE_TABLE).count()}")
print(f"   Aircraft count: {spark.table(BRONZE_TABLE).select(F.size('states')).collect()[0][0]}")
print(f"\n🏁 Notebook 2 Complete!")

🔍 Bronze Table Preview:
+----------+--------------------------+----------------------+------------------------------------------------------------+
|      time|               insert_dttm|           inserted_by|                                                 source_file|
+----------+--------------------------+----------------------+------------------------------------------------------------+
|1778003050|2026-05-05 18:00:57.763305|pendemajay94@gmail.com|dbfs:/Volumes/dev_team/testing/files_volume/Ajay/ajay_fli...|
+----------+--------------------------+----------------------+------------------------------------------------------------+


📊 Quick stats:
   Total rows    : 1
   Aircraft count: 10676

🏁 Notebook 2 Complete!
